# TrustBench v2 — EDA notebook

End-to-end exploratory analysis of LLM trust responses (Claude Opus 4.7, GPT-5.5, Gemini 3.1) against WVS-7 human data.

Pipeline modules: `loader.py`, `scoring.py`, `aggregate.py`, `wvs.py`, `figures.py`.
Figures are saved to `figures/eda/`. The narrative writeup lives in `figures/eda/EDA_report.md`.

In [ ]:
from __future__ import annotations
import sys, pathlib
REPO = pathlib.Path.cwd()
while not (REPO / 'data').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from src.analysis.loader   import load_items, load_llm_results, load_wvs_country_means, MODEL_ORDER
from src.analysis.scoring  import add_trust_score, wvs_country_trust
from src.analysis.aggregate import (model_item_means, model_section_means,
                                    framing_robustness, cross_model_item_matrix,
                                    refusal_rates)
from src.analysis.wvs       import wvs_summary, country_level_trust, llm_country_resemblance
from src.analysis.figures   import make_all
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)

## 1 · Load data

`load_llm_results('all')` pulls the union of base + verbal jsonl files for the three models (4 500 rows / model = 13 500 total).
`load_items()` parses `data/trust_items_v2.csv` and synthesizes the WVS column name (`Q64P`, `Q292A`, ...) for each item.

In [ ]:
items = load_items()
llm   = load_llm_results('all')
wvs_means = load_wvs_country_means()

print('items :', items.shape, '— sections:', items['section'].value_counts().to_dict())
print('llm   :', llm.shape, '— models:', list(llm['model'].unique()))
print('wvs   :', wvs_means.shape, '— countries with Q292:', wvs_means['Q292A'].notna().sum())
llm.head(3)

## 2 · Score responses

`add_trust_score` normalises every response to a 0–1 trust score (0 = no trust, 1 = full trust). It also runs a recovery parser on responses where the leading-token parser failed but a choice is still extractable from the response text.

In [ ]:
scored = add_trust_score(llm, items)
print(scored.groupby(['model','parsed_source']).size().unstack(fill_value=0))
print()
print('Refusal rate (%):')
print(refusal_rates(scored, ['model','framing_label'])
      .pivot(index='model', columns='framing_label', values='refusal_pct').round(2))

## 3 · Section-level trust by model, with WVS pooled overlay

In [ ]:
sec = model_section_means(scored)
wsum = wvs_summary(items).merge(items[['id','section']], left_on='item_id', right_on='id')
wvs_sec = wsum.groupby('section')['wvs_mean'].mean().rename('wvs_pooled')
summary = sec.pivot(index='section', columns='model', values='trust_mean').join(wvs_sec)
summary.round(3)

## 4 · Per-item calibration to WVS pooled mean

In [ ]:
mat = cross_model_item_matrix(scored)
wsum2 = wvs_summary(items).set_index('item_id')
calib_rows = []
for model in MODEL_ORDER:
    sub = mat[model].dropna()
    iids = sub.index.intersection(wsum2.index)
    x = wsum2.loc[iids, 'wvs_mean'].values
    y = sub.loc[iids].values
    rho, _ = spearmanr(x, y)
    calib_rows.append({'model': model, 'n_items': len(iids),
                       'spearman_rho': rho,
                       'rmse': float(np.sqrt(((x - y) ** 2).mean())),
                       'bias_llm_minus_wvs': float((y - x).mean())})
pd.DataFrame(calib_rows).round(3)

## 5 · Largest discrepancies (LLM − WVS pooled)

In [ ]:
disc = mat.sub(wsum2['wvs_mean'], axis=0).dropna(how='all')
items_lab = items.set_index('id')
for model in MODEL_ORDER:
    s = disc[model].dropna().sort_values()
    print(f'\n=== {model} ===')
    print('  Most UNDER-trusting (LLM − WVS most negative):')
    for iid, v in s.head(5).items():
        lab = items_lab.loc[iid, 'institution'] or items_lab.loc[iid, 'statement'] or iid
        print(f'    {v:+.2f}   {lab}')
    print('  Most OVER-trusting (LLM − WVS most positive):')
    for iid, v in s.tail(5).items():
        lab = items_lab.loc[iid, 'institution'] or items_lab.loc[iid, 'statement'] or iid
        print(f'    {v:+.2f}   {lab}')

## 6 · Framing robustness (numeric-original vs reversed vs verbal)

In [ ]:
fr = framing_robustness(scored)
wide = fr.pivot_table(index=['model','item_id'], columns='framing_label',
                       values='trust_mean').reset_index()
for model in MODEL_ORDER:
    sub = wide[wide['model']==model].dropna(subset=['numeric-original','numeric-reversed'])
    r1 = sub[['numeric-original','numeric-reversed']].corr().iloc[0,1]
    sub2 = wide[wide['model']==model].dropna(subset=['numeric-original','verbal'])
    r2 = sub2[['numeric-original','verbal']].corr().iloc[0,1]
    print(f'{model:18s}  r(num↔reversed) = {r1:.3f}   r(num↔verbal) = {r2:.3f}')

## 7 · Cross-model agreement

In [ ]:
pairs = [('Claude Opus 4.7','GPT-5.5'), ('Claude Opus 4.7','Gemini 3.1'), ('GPT-5.5','Gemini 3.1')]
for a, b in pairs:
    sub = mat[[a,b]].dropna()
    rho, _ = spearmanr(sub[a], sub[b])
    rmse = float(np.sqrt(((sub[a]-sub[b])**2).mean()))
    print(f'{a:18s} vs {b:18s}  ρ={rho:.3f}  RMSE={rmse:.3f}')

## 8 · LLM ↔ WVS country resemblance (Q64–Q89)

In [ ]:
res = llm_country_resemblance(scored, items, use_section='wvs_confidence')
for model in MODEL_ORDER:
    print(f'\n{model} — top-5 closest countries by Spearman ρ:')
    print(res[res['model']==model].sort_values('spearman', ascending=False).head(5)
            [['country','spearman','rmse','n_items']].to_string(index=False))

## 9 · Generate all publication figures

Writes PNG (300 dpi) + PDF for each figure into `figures/eda/`.

In [ ]:
make_all(scored, items)
import os
for f in sorted(os.listdir(REPO / 'figures' / 'eda')):
    path = REPO / 'figures' / 'eda' / f
    print(f'  {f}  {path.stat().st_size // 1024} KB')

---
**Narrative findings** are in [`figures/eda/EDA_report.md`](../../figures/eda/EDA_report.md).